In [19]:
"""
NB01_Spherical_Harmonics_Gravity_Evaluator (Moon)

Purpose
-------
1) Download / cache the GRGM900C high-degree lunar gravity model (PDS3 SHA format).
2) Parse fully-normalized C_lm, S_lm coefficients.
3) Evaluate Moon-fixed spherical-harmonic gravity acceleration [km/s^2].
4) Provide sanity checks (degree convergence, polar test, frame rotation).

Key design choices
-------------------
- Fully-normalized ALF recursion computed directly (geodetic convention,
  NO Condon-Shortley phase) to match GRAIL coefficient convention.
- Polar-safe longitude acceleration via limiting form (avoids 1/cos(phi)
  singularity for the polar orbit in NB02).
- Inner summation loops accelerated with Numba JIT where available;
  falls back to pure Python if Numba is not installed.

Dependencies: numpy, math, pathlib, urllib (std lib), [optional] numba
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import urllib.request
import numpy as np
import math
from datetime import datetime, timezone

# --- Optional Numba ---
try:
    from numba import njit
    HAS_NUMBA = True
except ImportError:
    HAS_NUMBA = False
    # identity decorator so code still runs without Numba
    def njit(*args, **kwargs):
        def wrapper(fn):
            return fn
        if len(args) == 1 and callable(args[0]):
            return args[0]
        return wrapper

print("Numba available:", HAS_NUMBA)

# --- Project folder ---
MODEL_DIR = Path("./gravity_models")
MODEL_DIR.mkdir(exist_ok=True)

# --- PDS product: GRGM900C coefficients ---
PDS_SHA_URL  = "https://pds-geosciences.wustl.edu/grail/grail-l-lgrs-5-rdr-v1/grail_1001/shadr/gggrx_0900c_sha.tab"
LOCAL_SHA_PATH = MODEL_DIR / "gggrx_0900c_sha.tab"

Numba available: True


In [20]:
# ============================================================
# Cell 1 : Download / cache gravity coefficient file
# ============================================================
def http_get_bytes(url: str, timeout: int = 240) -> bytes:
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 Python-urllib", "Accept": "*/*"},
    )
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()


def download_if_missing(url: str, dst: Path, min_bytes: int = 1_000_000) -> Path:
    if dst.exists() and dst.stat().st_size >= min_bytes:
        print(f"Using cached: {dst}  ({dst.stat().st_size/1e6:.1f} MB)")
        return dst
    data = http_get_bytes(url)
    if len(data) < min_bytes:
        raise RuntimeError(f"Downloaded too small ({len(data)} bytes). Likely an error page.")
    dst.write_bytes(data)
    print(f"Saved: {dst}  ({dst.stat().st_size/1e6:.1f} MB)")
    return dst


sha_path = download_if_missing(PDS_SHA_URL, LOCAL_SHA_PATH)

Using cached: gravity_models\gggrx_0900c_sha.tab  (49.6 MB)


In [21]:
# ============================================================
# Cell 2 : Parse PDS3 SHA table -> SHModel dataclass
# ============================================================
@dataclass
class SHModel:
    C: np.ndarray            # (Lmax+1, Lmax+1) fully-normalised Clm
    S: np.ndarray            # (Lmax+1, Lmax+1) fully-normalised Slm
    sigmaC: np.ndarray
    sigmaS: np.ndarray
    Lmax: int
    mu_km3_s2: float         # GM from the model header
    Rref_km: float           # reference radius from the model header
    normalized: bool
    source: str


def parse_gggrx_pds3_fixed(path: Path) -> SHModel:
    """Parse the fixed-width GGGRX SHA table into an SHModel."""
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    if len(lines) < 3:
        raise ValueError("File too short; expected header + blank + coefficients.")

    # Header record (record 1) is comma-delimited
    parts = [p.strip() for p in lines[0].split(",") if p.strip()]
    if len(parts) < 6:
        raise ValueError("Unexpected header format.")

    Rref_km   = float(parts[0])
    mu_km3_s2 = float(parts[1])
    Lmax      = int(float(parts[3]))
    normalized = (int(float(parts[5])) == 1)

    C      = np.zeros((Lmax + 1, Lmax + 1), dtype=float)
    S      = np.zeros((Lmax + 1, Lmax + 1), dtype=float)
    sigmaC = np.zeros((Lmax + 1, Lmax + 1), dtype=float)
    sigmaS = np.zeros((Lmax + 1, Lmax + 1), dtype=float)

    for ln in lines[2:]:
        if not ln.strip() or len(ln) < 107:
            continue
        try:
            l  = int(ln[0:5])
            m  = int(ln[6:11])
            c  = float(ln[12:35])
            s  = float(ln[36:59])
            sc = float(ln[60:83])
            ss = float(ln[84:107])
        except Exception:
            continue
        if 0 <= m <= l <= Lmax:
            C[l, m] = c;  S[l, m] = s
            sigmaC[l, m] = sc;  sigmaS[l, m] = ss

    if C[0, 0] == 0.0:
        C[0, 0] = 1.0          # implicit C00 = 1

    return SHModel(
        C=C, S=S, sigmaC=sigmaC, sigmaS=sigmaS,
        Lmax=Lmax, mu_km3_s2=mu_km3_s2, Rref_km=Rref_km,
        normalized=normalized, source=str(path),
    )


sh = parse_gggrx_pds3_fixed(sha_path)

print("Loaded gravity model:")
print(f"  source        = {sh.source}")
print(f"  Lmax          = {sh.Lmax}")
print(f"  normalized    = {sh.normalized}")
print(f"  Rref [km]     = {sh.Rref_km}")
print(f"  GM [km^3/s^2] = {sh.mu_km3_s2}")
print(f"  C00           = {sh.C[0,0]}")
print(f"  C20           = {sh.C[2,0]}")

# --- Consistency note ---
# The model's GM and Rref may differ slightly from NB00 constants.
# The gravity evaluator uses sh.mu_km3_s2 and sh.Rref_km internally,
# while NB00's mu/R_moon are used for orbit geometry.  This is standard
# practice (evaluator uses model-native constants).
try:
    _nb00_mu = CONFIG["mu_km3_s2"]
    print(f"\n  NB00 mu       = {_nb00_mu}")
    print(f"  Model GM      = {sh.mu_km3_s2}")
    print(f"  Difference    = {abs(sh.mu_km3_s2 - _nb00_mu):.6e} km^3/s^2")
except NameError:
    print("\n  (CONFIG not loaded — run NB00 first for cross-check)")

Loaded gravity model:
  source        = gravity_models\gggrx_0900c_sha.tab
  Lmax          = 900
  normalized    = True
  Rref [km]     = 1738.0
  GM [km^3/s^2] = 4902.79996708864
  C00           = 1.0
  C20           = -9.08866163613439e-05

  (CONFIG not loaded — run NB00 first for cross-check)


In [22]:
# ============================================================
# Cell 3 : (Optional) Write ICGEM .gfc for interoperability
# ============================================================
def write_icgem_gfc(sh: SHModel, out_path: Path,
                     modelname: str = "GRGM900C_GGGRX_0900C") -> Path:
    lines = []
    lines.append("begin_of_head")
    lines.append(f"modelname {modelname}")
    lines.append("product_type gravity_field")
    lines.append("format icgem")
    lines.append("errors no")
    lines.append(f"norm {'fully_normalized' if sh.normalized else 'unnormalized'}")
    lines.append(f"max_degree {sh.Lmax}")
    lines.append(f"earth_gravity_constant {sh.mu_km3_s2 * 1e9:.11f}")  # km^3/s^2 -> m^3/s^2
    lines.append(f"radius {sh.Rref_km * 1000.0:.1f}")                  # km -> m
    lines.append("tide_system zero_tide")
    lines.append(f"epoch {datetime.now(timezone.utc).strftime('%Y-%m-%d')}")
    lines.append("end_of_head")

    for l in range(sh.Lmax + 1):
        for m in range(l + 1):
            lines.append(f"gfc {l:4d} {m:4d} {sh.C[l,m]: .16e} {sh.S[l,m]: .16e}")

    out_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return out_path


gfc_path = MODEL_DIR / "GRGM900C_from_PDS.gfc"
write_icgem_gfc(sh, gfc_path)
print(f"Wrote: {gfc_path}  ({gfc_path.stat().st_size/1e6:.1f} MB)")

Wrote: gravity_models\GRGM900C_from_PDS.gfc  (25.6 MB)


In [23]:
# ============================================================
# Cell 4 : Fully-Normalised ALF Recursion (geodetic, no CS phase)
# ============================================================
#
# Geodetic convention for fully-normalised associated Legendre functions:
#
#   Pbar_{l,m}(x) = N_{l,m} * P_{l,m}(x)
#
# where P_{l,m}(x) is the POSITIVE-definite (no Condon-Shortley phase)
# unnormalised associated Legendre function, and
#
#   N_{l,0} = sqrt(2l+1)
#   N_{l,m} = sqrt( 2(2l+1) (l-m)! / (l+m)! ),  m > 0
#
# Rather than building unnormalised P first and then normalising (which
# overflows for high l), we recurse directly on Pbar using the standard
# normalised recursion coefficients.  This is numerically stable to
# degree ~2700 in float64.
#
# References:
#   Holmes & Featherstone (2002), J. Geodesy 76, 279-299
#   Montenbruck & Gill, Satellite Orbits, §3.2

@njit(cache=True)
def _fully_normalized_legendre(sin_phi: float, cos_phi: float, L: int):
    """
    Compute fully-normalised ALFs  Pbar[l,m](sin_phi)  and their
    derivative  dPbar_dphi[l,m]  w.r.t. geocentric latitude phi,
    for l = 0..L, m = 0..l.

    Uses the direct normalised recursion (no Condon-Shortley phase).

    Parameters
    ----------
    sin_phi, cos_phi : float   sin/cos of geocentric latitude
    L : int                    maximum degree

    Returns
    -------
    Pbar       : (L+1, L+1) float64
    dPbar_dphi : (L+1, L+1) float64
    """
    Pbar  = np.zeros((L + 1, L + 1))
    dPbar = np.zeros((L + 1, L + 1))

    # --- Sectoral seed: Pbar[0,0] = 1 ---
    Pbar[0, 0] = 1.0
    dPbar[0, 0] = 0.0

    # --- Diagonal (sectoral) recursion: Pbar[m,m] ---
    # Pbar[m,m] = cos_phi * sqrt((2m+1) / (2m)) * Pbar[m-1,m-1]
    for m in range(1, L + 1):
        fac = math.sqrt((2.0 * m + 1.0) / (2.0 * m))
        Pbar[m, m]  = cos_phi * fac * Pbar[m - 1, m - 1]
        # derivative by product rule:
        # d/dphi [cos_phi * f * Pbar[m-1,m-1]]
        #   = -sin_phi * f * Pbar[m-1,m-1]  +  cos_phi * f * dPbar[m-1,m-1]
        dPbar[m, m] = fac * (-sin_phi * Pbar[m - 1, m - 1]
                             + cos_phi * dPbar[m - 1, m - 1])

    # --- First sub-diagonal: Pbar[m+1, m] ---
    # Pbar[m+1,m] = sin_phi * sqrt(2m+3) * Pbar[m,m]
    for m in range(0, L):
        fac = math.sqrt(2.0 * m + 3.0)
        Pbar[m + 1, m]  = sin_phi * fac * Pbar[m, m]
        dPbar[m + 1, m] = fac * (cos_phi * Pbar[m, m]
                                 + sin_phi * dPbar[m, m])

    # --- General two-term recursion for l >= m+2 ---
    # Pbar[l,m] = alpha * sin_phi * Pbar[l-1,m]  -  beta * Pbar[l-2,m]
    for m in range(0, L + 1):
        for l in range(m + 2, L + 1):
            ls = float(l)
            ms = float(m)
            alpha = math.sqrt((4.0*ls*ls - 1.0) / (ls*ls - ms*ms))
            beta  = math.sqrt(((ls - 1.0)**2 - ms*ms) / (4.0*(ls - 1.0)**2 - 1.0))

            Pbar[l, m]  = alpha * (sin_phi * Pbar[l-1, m] - beta * Pbar[l-2, m])
            dPbar[l, m] = alpha * (cos_phi * Pbar[l-1, m]
                                   + sin_phi * dPbar[l-1, m]
                                   - beta * dPbar[l-2, m])

    return Pbar, dPbar


# Quick smoke test
_P, _dP = _fully_normalized_legendre(0.0, 1.0, 4)
print("Pbar(sin=0, L=4) diagonal:")
for m in range(5):
    print(f"  Pbar[{m},{m}] = {_P[m,m]:.10f}")
# Expected: Pbar[0,0]=1, Pbar[1,1]=sqrt(3)*1 = 1.7320508...
#           Pbar[2,2]=sqrt(5/4)*sqrt(3)=... etc.
print(f"  Pbar[2,0] = {_P[2,0]:.10f}  (expect -sqrt(5)/2 = {-math.sqrt(5)/2:.10f})")

Pbar(sin=0, L=4) diagonal:
  Pbar[0,0] = 1.0000000000
  Pbar[1,1] = 1.2247448714
  Pbar[2,2] = 1.3693063938
  Pbar[3,3] = 1.4790199458
  Pbar[4,4] = 1.5687375498
  Pbar[2,0] = -1.1180339887  (expect -sqrt(5)/2 = -1.1180339887)


In [24]:
# ============================================================
# Cell 5 : Gravity Acceleration Evaluator (spherical harmonic)
# ============================================================
#
# Evaluates the TOTAL gravitational acceleration (including l=0
# central term) in the Moon body-fixed Cartesian frame.
#
# The disturbing (non-central) acceleration needed for the
# encounter framework is obtained downstream as:
#   dg = gravity_accel_sph_harm(sh, r, L) - (-mu/r^3)*r
#
# Polar handling
# --------------
# The east-component  a_lambda = -(1/(r cos phi)) dV/dlambda
# is singular at the poles.  We use the identity
#   Pbar_{l,m}(sin phi) / cos phi  ->  finite limit
# via the derivative recursion (Pines approach).  As a practical
# and transparent alternative, we evaluate at a clamped latitude
# |phi| <= pi/2 - eps_pole and document the residual error.
# At 50 km altitude the along-track distance corresponding to
# eps_pole ~ 1e-10 rad is ~0.18 mm, far below any physical scale.

EPS_POLE = 1e-10   # latitude clamp offset from pole [rad]


def gravity_accel_sph_harm(
    sh: SHModel,
    r_bf_km: np.ndarray,
    degree: int = 50,
) -> np.ndarray:
    """
    Total gravitational acceleration in Moon body-fixed frame [km/s^2].

    Parameters
    ----------
    sh       : SHModel with fully-normalised C, S coefficients
    r_bf_km  : (3,) position in body-fixed frame [km]
    degree   : max SH degree to include (clamped to sh.Lmax)

    Returns
    -------
    a_bf : (3,) acceleration [km/s^2] in body-fixed Cartesian
    """
    x, y, z = float(r_bf_km[0]), float(r_bf_km[1]), float(r_bf_km[2])
    r = math.sqrt(x*x + y*y + z*z)
    if r == 0.0:
        raise ValueError("Position at centre of body.")

    # Geocentric latitude phi, longitude lambda
    lam     = math.atan2(y, x)
    sin_phi = z / r
    cos_phi = math.sqrt(max(0.0, 1.0 - sin_phi * sin_phi))

    # Clamp latitude away from exact pole for lambda-derivative stability
    if cos_phi < EPS_POLE:
        # Shift sin_phi so that cos_phi = EPS_POLE
        sin_phi = math.copysign(math.sqrt(1.0 - EPS_POLE**2), sin_phi)
        cos_phi = EPS_POLE

    L = min(int(degree), sh.Lmax)

    Pbar, dPbar_dphi = _fully_normalized_legendre(sin_phi, cos_phi, L)

    # --- Call the inner summation (separate function for Numba) ---
    ar, aphi, alam = _sh_summation(
        sh.C, sh.S, Pbar, dPbar_dphi,
        L, sh.mu_km3_s2, sh.Rref_km,
        r, lam, sin_phi, cos_phi,
    )

    # --- Convert spherical (r, phi, lambda) to body-fixed Cartesian ---
    cos_lam = math.cos(lam)
    sin_lam = math.sin(lam)

    ax = (ar * cos_phi * cos_lam
          + aphi * (-sin_phi) * cos_lam
          + alam * (-sin_lam))
    ay = (ar * cos_phi * sin_lam
          + aphi * (-sin_phi) * sin_lam
          + alam * cos_lam)
    az = (ar * sin_phi
          + aphi * cos_phi)
    # alam * 0 for z-component

    return np.array([ax, ay, az], dtype=np.float64)


@njit(cache=True)
def _sh_summation(
    C, S, Pbar, dPbar_dphi,
    L, mu, a_ref,
    r, lam, sin_phi, cos_phi,
):
    """
    Inner double-loop summation for SH gravity.
    Returns spherical acceleration components (ar, aphi, alam).
    """
    # Precompute cos(m*lam), sin(m*lam) via recurrence
    cosml = np.empty(L + 1)
    sinml = np.empty(L + 1)
    cosml[0] = 1.0
    sinml[0] = 0.0
    c1 = math.cos(lam)
    s1 = math.sin(lam)
    for m in range(1, L + 1):
        cosml[m] = cosml[m - 1] * c1 - sinml[m - 1] * s1
        sinml[m] = sinml[m - 1] * c1 + cosml[m - 1] * s1

    rho = a_ref / r

    ar_acc   = 0.0
    aphi_acc = 0.0
    alam_acc = 0.0

    rho_l = 1.0   # (a/r)^0 for l=0
    for l in range(0, L + 1):
        if l > 0:
            rho_l *= rho

        sum_V      = 0.0
        sum_dV_phi = 0.0
        sum_dV_lam = 0.0

        for m in range(0, l + 1):
            trig = C[l, m] * cosml[m] + S[l, m] * sinml[m]

            sum_V      += Pbar[l, m] * trig
            sum_dV_phi += dPbar_dphi[l, m] * trig

            if m > 0:
                dtrig = m * (-C[l, m] * sinml[m] + S[l, m] * cosml[m])
                sum_dV_lam += Pbar[l, m] * dtrig

        # V_l = (mu/r) * rho_l * sum_V
        # dV/dr  = -(mu/r^2) * (l+1) * rho_l * sum_V
        # dV/dphi = (mu/r) * rho_l * sum_dV_phi
        # dV/dlam = (mu/r) * rho_l * sum_dV_lam
        ar_acc   += (l + 1) * rho_l * sum_V
        aphi_acc += rho_l * sum_dV_phi
        alam_acc += rho_l * sum_dV_lam

    # Spherical acceleration components:
    #   a_r     = -dV/dr         = (mu/r^2) * ar_acc  ... wait, sign:
    #   V = (mu/r) Sigma => dV/dr = -(mu/r^2)(l+1) rho_l sum_V
    #   a_r = -dV/dr = +(mu/r^2) Sigma (l+1) rho_l sum_V
    #   ... but gravity points inward, so the code's original convention
    #   a_r = -(mu/r^2) * ar_acc  gives the correct INWARD acceleration
    #   because ar_acc includes the l=0 term which gives a_r ~ -mu/r^2 (inward).
    #
    # Let's be explicit:  a_r = -dV/dr.  V = mu/r sum => dV/dr|_{l=0} = -mu/r^2
    # so a_r|_{l=0} = +mu/r^2 ... that points outward.  No — the potential is
    # V = -mu/r for gravity (attractive).  But geodetic convention uses V = +mu/r.
    # The acceleration is then  g = +grad(V) for the geodetic potential,
    # which gives g_r = dV/dr = -(mu/r^2)(l+1)rho_l sum_V   (with l=0 giving -mu/r^2, inward).
    #
    # So:  a_r = -(mu/r^2) * ar_acc   is correct (negative = inward for l=0).
    r2 = r * r
    ar_out   = -(mu / r2) * ar_acc
    aphi_out = -(mu / r2) * aphi_acc
    alam_out = -(mu / r2) * (alam_acc / cos_phi)

    return ar_out, aphi_out, alam_out


print("gravity_accel_sph_harm() defined.")
if HAS_NUMBA:
    print("(Numba JIT will compile on first call.)")

gravity_accel_sph_harm() defined.
(Numba JIT will compile on first call.)


In [25]:
# ============================================================
# Cell 6 : Quick Sanity Tests
# ============================================================

# --- Test 1: equatorial point (lon=0, lat=0) ---
r_eq = np.array([sh.Rref_km + 50.0, 0.0, 0.0], dtype=float)
a2_eq  = gravity_accel_sph_harm(sh, r_eq, degree=2)
a50_eq = gravity_accel_sph_harm(sh, r_eq, degree=50)

r_mag_eq = np.linalg.norm(r_eq)
a_cent = sh.mu_km3_s2 / (r_mag_eq**2)

print("=== Test 1: Equatorial point (lon=0, lat=0) ===")
print(f"  r [km]            = {r_mag_eq}")
print(f"  central mu/r^2    = {a_cent:.10e}")
print(f"  deg=2  a [km/s^2] = {a2_eq}   |a| = {np.linalg.norm(a2_eq):.10e}")
print(f"  deg=50 a [km/s^2] = {a50_eq}  |a| = {np.linalg.norm(a50_eq):.10e}")
print(f"  |a|/central       = {np.linalg.norm(a50_eq)/a_cent:.10f}  (expect ~1.000)")

# --- Test 2: polar point (lat=+90) ---
r_pole = np.array([0.0, 0.0, sh.Rref_km + 50.0], dtype=float)
a50_pole = gravity_accel_sph_harm(sh, r_pole, degree=50)

print("\n=== Test 2: North polar point (lat=90) ===")
print(f"  r [km]            = {np.linalg.norm(r_pole)}")
print(f"  deg=50 a [km/s^2] = {a50_pole}")
print(f"  |a|               = {np.linalg.norm(a50_pole):.10e}")
print(f"  (should be finite and dominated by z-component)")

# --- Test 3: off-equator point (lat ~45 deg) ---
r_45 = (sh.Rref_km + 50.0) * np.array([1.0, 0.0, 1.0]) / np.sqrt(2.0)
a50_45 = gravity_accel_sph_harm(sh, r_45, degree=50)

print("\n=== Test 3: Mid-latitude point (lat ~45, lon=0) ===")
print(f"  r [km]            = {np.linalg.norm(r_45):.6f}")
print(f"  deg=50 a [km/s^2] = {a50_45}")
print(f"  |a|               = {np.linalg.norm(a50_45):.10e}")

=== Test 1: Equatorial point (lon=0, lat=0) ===
  r [km]            = 1788.0
  central mu/r^2    = 1.5335895678e-03
  deg=2  a [km/s^2] = [-1.53423768e-03 -4.32880328e-13  7.73656720e-13]   |a| = 1.5342376818e-03
  deg=50 a [km/s^2] = [-1.53441440e-03 -6.88866372e-08 -2.04760202e-07]  |a| = 1.5344144122e-03
  |a|/central       = 1.0005378521  (expect ~1.000)

=== Test 2: North polar point (lat=90) ===
  r [km]            = 1788.0
  deg=50 a [km/s^2] = [-4.32344335e-07 -1.38265675e-07 -1.53273404e-03]
  |a|               = 1.5327341092e-03
  (should be finite and dominated by z-component)

=== Test 3: Mid-latitude point (lat ~45, lon=0) ===
  r [km]            = 1788.000000
  deg=50 a [km/s^2] = [-1.08451399e-03  2.99280964e-07 -1.08347890e-03]
  |a|               = 1.5330026727e-03


In [26]:
# ============================================================
# Cell 7 : Degree Convergence Check
# ============================================================
def check_degree_convergence(sh, alt_km=50.0, degrees=(2, 10, 50, 150),
                              npts=6, seed=0):
    """Evaluate gravity at random directions for several max-degrees."""
    rng = np.random.default_rng(seed)
    r0 = sh.Rref_km + alt_km

    dirs = rng.normal(size=(npts, 3))
    dirs /= np.linalg.norm(dirs, axis=1)[:, None]
    pts = r0 * dirs

    results = []
    for i, rvec in enumerate(pts):
        row = {"i": i, "r_km": float(np.linalg.norm(rvec))}
        a_prev = None
        prevL = None
        for L in degrees:
            a = gravity_accel_sph_harm(sh, rvec, degree=L)
            row[f"|a|_L{L}"] = float(np.linalg.norm(a))
            if a_prev is not None:
                row[f"da_{prevL}->{L}"] = float(np.linalg.norm(a - a_prev))
            a_prev = a
            prevL = L
        results.append(row)
    return results


table = check_degree_convergence(sh, alt_km=50.0, degrees=(2, 10, 50, 150), npts=6)
for row in table:
    print(row)

{'i': 0, 'r_km': 1788.0, '|a|_L2': 0.001532804773683018, '|a|_L10': 0.001533215359628233, 'da_2->10': 4.5809300104540226e-07, '|a|_L50': 0.001533398881816392, 'da_10->50': 2.3285198478990224e-07, '|a|_L150': 0.0015334172025774661, 'da_50->150': 1.994230994456092e-08}
{'i': 1, 'r_km': 1788.0, '|a|_L2': 0.0015334943057548577, '|a|_L10': 0.0015336637232731558, 'da_2->10': 2.0127258657236124e-07, '|a|_L50': 0.0015336421060465277, 'da_10->50': 8.207482025653718e-08, '|a|_L150': 0.0015336179563604103, 'da_50->150': 4.376267066233632e-08}
{'i': 2, 'r_km': 1788.0, '|a|_L2': 0.0015338727425743466, '|a|_L10': 0.0015341071152599778, 'da_2->10': 2.915069863106995e-07, '|a|_L50': 0.0015338050480524672, 'da_10->50': 4.3414439407365137e-07, '|a|_L150': 0.001533817024310763, 'da_50->150': 2.7807320248055155e-08}
{'i': 3, 'r_km': 1788.0000000000002, '|a|_L2': 0.001534155852494491, '|a|_L10': 0.0015348268068284747, 'da_2->10': 6.750053819556195e-07, '|a|_L50': 0.0015348106152822786, 'da_10->50': 6.97744

In [27]:
# ============================================================
# Cell 8 : Frame-Rotation Sensitivity Check
# ============================================================
def vec_lon_lat(r_km: np.ndarray):
    x, y, z = float(r_km[0]), float(r_km[1]), float(r_km[2])
    r = math.sqrt(x*x + y*y + z*z)
    lon = math.degrees(math.atan2(y, x))
    lat = math.degrees(math.asin(max(-1.0, min(1.0, z / r))))
    return lon, lat, r


def rotate_about_z(r_km: np.ndarray, deg: float) -> np.ndarray:
    th = math.radians(deg)
    c, s = math.cos(th), math.sin(th)
    R = np.array([[ c, -s, 0.0],
                  [ s,  c, 0.0],
                  [0.0, 0.0, 1.0]], dtype=float)
    return R @ r_km


def state_frame_check_single(sh, r_km, degree=50, test_deg=30.0):
    a0    = gravity_accel_sph_harm(sh, r_km, degree=degree)
    r_rot = rotate_about_z(r_km, test_deg)
    a_rot = gravity_accel_sph_harm(sh, r_rot, degree=degree)

    lon0, lat0, rr0 = vec_lon_lat(r_km)
    lon1, lat1, rr1 = vec_lon_lat(r_rot)

    print("Input position:")
    print(f"  r [km] = {r_km}  |r| = {rr0:.3f}")
    print(f"  lon,lat [deg] = ({lon0:.3f}, {lat0:.3f})")
    print(f"Rotated +{test_deg} deg about Z:")
    print(f"  lon,lat [deg] = ({lon1:.3f}, {lat1:.3f})  |r| = {rr1:.3f}")
    print(f"\na at original : {a0}   |a| = {np.linalg.norm(a0):.10e}")
    print(f"a at rotated  : {a_rot}   |a| = {np.linalg.norm(a_rot):.10e}")
    da = np.linalg.norm(a_rot - a0)
    print(f"\n|a_rot - a0|  = {da:.6e}")
    print(f"relative diff = {da / np.linalg.norm(a0):.4f}")


r_test = np.array([sh.Rref_km + 50.0, 0.0, 0.0], dtype=float)
state_frame_check_single(sh, r_test, degree=50, test_deg=30.0)

print("\n--- NB01 complete ---")

Input position:
  r [km] = [1788.    0.    0.]  |r| = 1788.000
  lon,lat [deg] = (0.000, 0.000)
Rotated +30.0 deg about Z:
  lon,lat [deg] = (30.000, 0.000)  |r| = 1788.000

a at original : [-1.53441440e-03 -6.88866372e-08 -2.04760202e-07]   |a| = 1.5344144122e-03
a at rotated  : [-1.32851789e-03 -7.66513177e-04  2.80783891e-07]   |a| = 1.5337869187e-03

|a_rot - a0|  = 7.936186e-04
relative diff = 0.5172

--- NB01 complete ---
